# Multi-Agency Ridership Data Standardization and Stop-Level Aggregation Workflow


This notebook processes ridership data from multiple transit agencies to create representative stop-level datasets for weekdays, Saturdays, and Sundays. For each agency, the workflow generally includes:

- Adding day type based on the service date (Weekday, Saturday, Sunday), with holidays normalized or removed as needed.
- Cleaning and renaming columns to ensure consistency across datasets (stop IDs, stop names, and ridership fields).
- Summing boardings and alightings across directions at each stop, and aggregating multiple time periods where applicable.
- Calculating weighted averages when ridership is reported over multiple periods, using the number of service days as weights.
- Setting a ridership basis flag to indicate that values represent reported daily boardings and/or alightings.
- Computing final average ridership per stop, and day type, producing clean, representative stop-level datasets ready for modeling, analysis, or comparison across agencies.

This approach standardizes ridership data across agencies with differing data formats, missing values, and service coverage, enabling consistent cross-agency analysis.

In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
from pandas.tseries.holiday import USFederalHolidayCalendar
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships'

In [4]:
bart_ridership = pd.read_excel(f"{GCS_FILE_PATH}/bart_ridership.xlsx")
big_blue_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/big_blue_bus_ridership.xlsx")
caltrain_ridership = pd.read_excel(f"{GCS_FILE_PATH}/caltrain_ridership.xlsx")
culver_citybus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/culver_citybus_ridership.xlsx")
foothill_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/foothill_transit_ridership.xlsx")
fresno_area_express_ridership = pd.read_excel(f"{GCS_FILE_PATH}/fresno_area_express_ridership.xlsx")
gold_coast_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/gold_coast_transit_ridership.xlsx")
golden_gate_park_shuttle_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_park_shuttle_ridership.xlsx")
golden_gate_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_transit_ridership.xlsx")
long_beach_transit_ridership=  pd.read_excel(f"{GCS_FILE_PATH}/long_beach_transit_ridership.xlsx")
octa_ridership = pd.read_excel(f"{GCS_FILE_PATH}/octa_ridership.xlsx")
omni_trans_ridership= pd.read_excel(f"{GCS_FILE_PATH}/omni_trans_ridership.xlsx")
riverside_transit_ridership= pd.read_excel(f"{GCS_FILE_PATH}/riverside_transit_ridership.xlsx")
sacrt_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sacrt_bus_ridership.xlsx")
samtrans_ridership = pd.read_excel(f"{GCS_FILE_PATH}/samtrans_ridership.xlsx")
santa_cruz_metro_ridership = pd.read_excel(f"{GCS_FILE_PATH}/santa_cruz_metro_ridership.xlsx")
sbmtd_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sbmtd_ridership.xlsx")
sdmts_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sdmts_ridership.xlsx")
sunline_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sunline_transit_ridership.xlsx")
burbank_ridership = pd.read_excel(f"{GCS_FILE_PATH}/burbank_ridership.xlsx")


In [5]:
def add_day_type(df, date_col, output_col='day_type'):
    """
    Classify each row as 'Weekday', 'Saturday', or 'Sunday' based on a specified date column.
    This is useful when daily datasets include labels such as 'Holiday' or when no day_type
    column exists. The function converts the date column, identifies the day of week, and
    standardizes the day-type classification accordingly. Either a start_date or end_date
    column can be used for datasets with daily reporting.
    """
    df = df.copy()

    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    dow = df[date_col].dt.dayofweek  

    df[output_col] = np.select(
        condlist=[dow.eq(5), dow.eq(6)],
        choicelist=['Saturday', 'Sunday'],
        default='Weekday'
    )

    return df

In [6]:
# Counting number of days without holidays
def count_service_days(row):
    
    if str(row.day_type).lower() == "all":
        return (row.end_date - row.start_date).days + 1

    dates = pd.date_range(row.start_date, row.end_date)

    if row.day_type == "Weekday":
        return sum(d.weekday() < 5 for d in dates)

    elif row.day_type == "Saturday":
        return sum(d.weekday() == 5 for d in dates)

    elif row.day_type == "Sunday":
        return sum(d.weekday() == 6 for d in dates)

In [7]:
# Counting number of days with holidays
def count_service_days_wo_holidays(row):
    dates = pd.date_range(row.start_date, row.end_date)

    if row.day_type == "Weekday":
        return sum((d.weekday() < 5) and (d not in holiday_set) for d in dates)

    elif row.day_type == "Saturday":
        return sum((d.weekday() == 5) and (d not in holiday_set) for d in dates)

    elif row.day_type == "Sunday":
        return sum((d.weekday() == 6) and (d not in holiday_set) for d in dates)

In [8]:
def compute_averages(df, ridership_cols, group_cols):
    """
    Computes average of one or more ridership columns per group in long format.

    Parameters:
    - df: pandas DataFrame
    - ridership_cols: list of columns to average (e.g., ['daily_boardings', 'daily_alightings'])
    - group_cols: list of columns to group by (e.g., ['stop_name', 'day_type'])

    Returns:
    - DataFrame with columns: group_cols + average of each ridership column
    """
    
    if isinstance(ridership_cols, str):
        ridership_cols = [ridership_cols]  # allow single string input
    
    averaged = df.groupby(group_cols)[ridership_cols].mean().reset_index()
    
    # rename each column to indicate it's averaged
    averaged = averaged.rename(columns={col: f"average_{col}" for col in ridership_cols})
    
    return averaged

In [9]:
master_cols = [
    "organization_name",
    "stop_id",
    "stop_name",
    "stop_lat",
    "stop_lon",
    "day_type",
    "average_daily_boardings",
    "average_daily_alightings",
    "start_date",
    "end_date"
]

def standardize_columns(df, master_cols):
    
    # add missing columns
    for col in master_cols:
        if col not in df.columns:
            df[col] = None

    # keep only master columns and order them
    return df[master_cols]

In [10]:
bart_ridership = add_day_type(bart_ridership, date_col='start_date')

bart_ridership = bart_ridership.rename(columns={
    'Station Name': 'stop_name',
    'Stop ID': 'stop_id',
    'Date': 'date'
})

# Grouping columns
group_cols = [
    "date", "stop_id", "stop_name",
    "start_date", "end_date", "day_type"
]

# Sum boardings and alightings, with explicit output names
total_bart_ridership = (
    bart_ridership
    .groupby(group_cols, as_index=False)
    .agg(
        daily_boardings=("Entries", "sum"),
        daily_alightings=("Exits", "sum")
    )
)

# Set the ridership basis flag
total_bart_ridership["daily_ridership_basis"] = "reported_daily"


In [11]:
# Calculate average ridership per day type and stop
total_bart_ridership['daily_boardings'] = pd.to_numeric(total_bart_ridership['daily_boardings'], errors='coerce')
total_bart_ridership['daily_alightings'] = pd.to_numeric(total_bart_ridership['daily_alightings'], errors='coerce')

average_bart_ridership = compute_averages(
    df=total_bart_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["stop_id", "day_type", "stop_name"]
)

average_bart_ridership["organization_name"] = "San Francisco Bay Area Rapid Transit District"

average_bart_ridership[['start_date','end_date']] = [bart_ridership['start_date'].min(), bart_ridership['end_date'].max()]
# Remove last digit from all stop_id
average_bart_ridership['stop_id'] = average_bart_ridership['stop_id'].astype(str).str[:-1]

bart_final = standardize_columns(average_bart_ridership, master_cols)

bart_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
14,San Francisco Bay Area Rapid Transit District,90116,Embarcadero,None,None,Weekday,15717.383142,17818.302682,2024-10-01,2025-09-30
112,San Francisco Bay Area Rapid Transit District,90520,West Dublin / Pleasanton,None,None,Sunday,515.057692,547.211538,2024-10-01,2025-09-30
53,San Francisco Bay Area Rapid Transit District,90250,Bayfair,None,None,Weekday,2554.141762,2598.915709,2024-10-01,2025-09-30
136,San Francisco Bay Area Rapid Transit District,90820,Pittsburg Center,None,None,Sunday,201.807692,234.153846,2024-10-01,2025-09-30
35,San Francisco Bay Area Rapid Transit District,90180,Balboa Park,None,None,Weekday,4304.513410,3979.992337,2024-10-01,2025-09-30


In [12]:
big_blue_bus_ridership.columns = big_blue_bus_ridership.columns.str.lower()
big_blue_bus_ridership['service_day'] = big_blue_bus_ridership['service_day'].str.strip().str.capitalize()


big_blue_bus_ridership = big_blue_bus_ridership.rename(columns={
    'service_day': 'day_type',
    'average_daily_boardings':'average_daily_boardings_period',
    'average_daily_alightings':'average_daily_alightings_period'    
})


# Sum boardings/alightings across directions within same stop & period
total_big_blue_bus_ridership = (
    big_blue_bus_ridership
    .groupby(['day_type',
              'stop_id','stop_name','stop_lat','stop_lon',
              'start_date','end_date'])
    .agg({
        'average_daily_boardings_period':'sum',
        'average_daily_alightings_period':'sum'
    })
    .reset_index()
)


In [13]:
big_blue_bus_ridership['start_date'] = pd.to_datetime(big_blue_bus_ridership['start_date'])
big_blue_bus_ridership['end_date'] = pd.to_datetime(big_blue_bus_ridership['end_date'])

total_big_blue_bus_ridership["n_days"] = total_big_blue_bus_ridership.apply(count_service_days, axis=1)

# Taking weighted average of the ridership across 4 different time periods
averaged_total_big_blue_bus_ridership = (
    total_big_blue_bus_ridership
    .groupby(['day_type', 'stop_name',
              'stop_id','stop_lat','stop_lon'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['average_daily_boardings_period'], weights=x['n_days']),
        'average_daily_alightings': np.average(x['average_daily_alightings_period'], weights=x['n_days'])
    }))
    .reset_index()
)


# Set the ridership basis flag
averaged_total_big_blue_bus_ridership["daily_ridership_basis"] = "reported_avg_daily"
averaged_total_big_blue_bus_ridership["organization_name"] = "Big Blue Bus"

averaged_total_big_blue_bus_ridership[['start_date','end_date']] = [big_blue_bus_ridership['start_date'].min(), big_blue_bus_ridership['end_date'].max()]

big_blue_bus_ridership_final = standardize_columns(averaged_total_big_blue_bus_ridership, master_cols)
big_blue_bus_ridership_final.sample(5)

/tmp/ipykernel_3806/1528211661.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
1905,Big Blue Bus,2520,MOTOR SB/GLENBARR NS,34.039474,-118.407149,Weekday,1.582946,0.783891,2024-08-01,2025-11-30
2270,Big Blue Bus,2008,VA HOSPITAL WB/BONSALL NS,34.053234,-118.453947,Weekday,22.273471,91.103463,2024-08-01,2025-11-30
1238,Big Blue Bus,1373,SANTA MONICA EB/26TH NS,34.033149,-118.474382,Sunday,32.549938,34.649911,2024-08-01,2025-11-30
220,Big Blue Bus,2207,LINCOLN NB/MINDANAO FS,33.980932,-118.439253,Saturday,66.824285,47.376000,2024-08-01,2025-11-30
816,Big Blue Bus,2426,BUNDY NB/MAYFIELD NS,34.047121,-118.468943,Sunday,1.006403,9.725680,2024-08-01,2025-11-30


In [14]:
caltrain_ridership = caltrain_ridership.rename(columns={
    'Origin Station': 'stop_name',
    'Date Type': 'day_type',
    'Average Ridership':'average_daily_boardings_period',  
})

caltrain_ridership = caltrain_ridership[
    caltrain_ridership["day_type"] != "Holiday"
]

# Getting number of days and using US Federal Holidays
start_date_caltrain = '2023-11-01'
end_date_caltrain = '2025-07-31'

#Getting all the federal holidays between the start and end date
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=start_date_caltrain, end=end_date_caltrain)

# Setting holidays to calculate the number of weekdays, saturday and sunday without holidays 
holiday_set = set(holidays)

caltrain_ridership["start_date"] = pd.to_datetime(caltrain_ridership["start_date"])
caltrain_ridership["end_date"] = pd.to_datetime(caltrain_ridership["end_date"])

caltrain_ridership["n_days"] = caltrain_ridership.apply(count_service_days_wo_holidays, axis=1)

In [15]:
# Taking weighted average of the ridership across 20 months time periods
averaged_caltrain_ridership = (
    caltrain_ridership
    .groupby(['day_type','stop_name'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['average_daily_boardings_period'], weights=x['n_days'])
    }))
    .reset_index()
)

# Set the ridership basis flag
averaged_caltrain_ridership["daily_ridership_basis"] = "reported_avg_daily"
averaged_caltrain_ridership["organization_name"] = "Caltrain"

averaged_caltrain_ridership[['start_date','end_date']] = [caltrain_ridership['start_date'].min(), caltrain_ridership['end_date'].max()]

caltrain_ridership_final = standardize_columns(averaged_caltrain_ridership, master_cols)

caltrain_ridership_final.sample(5)

/tmp/ipykernel_3806/3722569284.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
80,Caltrain,None,San Bruno,None,None,Weekday,312.963513,None,2023-11-01,2025-07-31
0,Caltrain,None,22nd Street,None,None,Saturday,433.679398,None,2023-11-01,2025-07-31
8,Caltrain,None,College Park,None,None,Saturday,2.835954,None,2023-11-01,2025-07-31
65,Caltrain,None,Burlingame,None,None,Weekday,542.005575,None,2023-11-01,2025-07-31
30,Caltrain,None,22nd Street,None,None,Sunday,354.256197,None,2023-11-01,2025-07-31


In [16]:
culver_citybus_ridership["daily_ridership_basis"] = "reported_avg_daily"



culver_citybus_ridership = culver_citybus_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
})

group_cols = [
     "stop_id", "stop_name", "day_type", "daily_ridership_basis"]

# Sum AVG On and AVG Off
sum_culver_citybus_ridership = (culver_citybus_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("AVG On", "sum"),
        average_daily_alightings=("AVG Off", "sum")
    )
)

sum_culver_citybus_ridership["organization_name"] = "Culver City Bus"

sum_culver_citybus_ridership[['start_date','end_date']] = [culver_citybus_ridership['start_date'].min(), culver_citybus_ridership['end_date'].max()]

culver_citybus_final = standardize_columns(sum_culver_citybus_ridership, master_cols)

culver_citybus_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
789,Culver City Bus,711,Culver Blvd/Braddock Dr,None,None,Weekday,3.7,1.1,2025-07-14,2025-08-25
333,Culver City Bus,333,Westwood Blvd/National Blvd,None,None,Saturday,5.5,11.2,2025-07-14,2025-08-25
746,Culver City Bus,675,Sepulveda Blvd/77th St,None,None,Sunday,7.6,14.6,2025-07-14,2025-08-25
646,Culver City Bus,639,Sepulveda Blvd/Santa Monica Blvd,None,None,Saturday,20.5,183.5,2025-07-14,2025-08-25
433,Culver City Bus,370,Overland Ave/Franklin Ave,None,None,Sunday,3.1,5.9,2025-07-14,2025-08-25


In [17]:
# Add day_type from the date column
foothill_transit_ridership = add_day_type(foothill_transit_ridership, date_col="date")

foothill_transit_ridership = foothill_transit_ridership.rename(columns={
    'stop_code': 'stop_id',
})

# Grouping columns
group_cols = [
    "date", "stop_id", "stop_lat", "stop_lon",
    "start_date", "end_date", "day_type"
]

# Sum boardings and alightings, with explicit output names
total_ridership_foothill = (
    foothill_transit_ridership
    .groupby(group_cols, as_index=False)
    .agg(
        daily_boardings=("boardings", "sum"),
        daily_alightings=("alightings", "sum")
    )
)

# Set the ridership basis flag
total_ridership_foothill["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_ridership_foothill['daily_boardings'] = pd.to_numeric(total_ridership_foothill['daily_boardings'], errors='coerce')
total_ridership_foothill['daily_alightings'] = pd.to_numeric(total_ridership_foothill['daily_alightings'], errors='coerce')

average_foothill_ridership = compute_averages(
    df=total_ridership_foothill,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=[ "stop_id", 
                "stop_lat", "stop_lon", "day_type"]
)

average_foothill_ridership["organization_name"] = "Foothill Transit"

average_foothill_ridership[['start_date','end_date']] = [foothill_transit_ridership['start_date'].min(), foothill_transit_ridership['end_date'].max()]
foothill_final = standardize_columns(average_foothill_ridership, master_cols)

foothill_final.sample(5)


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
37,Foothill Transit,24,None,34.035320,-117.919706,Sunday,5.211538,0.346154,2024-07-01,2025-06-30
1011,Foothill Transit,978,None,34.102724,-117.703063,Sunday,3.218750,0.156250,2024-07-01,2025-06-30
2577,Foothill Transit,1910,None,34.024014,-117.852012,Sunday,1.750000,0.222222,2024-07-01,2025-06-30
4435,Foothill Transit,3277,None,34.083551,-117.972918,Weekday,15.065134,2.689655,2024-07-01,2025-06-30
1158,Foothill Transit,1054,None,34.145775,-118.147144,Sunday,27.942308,0.653846,2024-07-01,2025-06-30


In [18]:
# Add day_type from the date column
fresno_area_express_ridership = add_day_type(fresno_area_express_ridership, date_col="Date")

fresno_area_express_ridership["daily_ridership_basis"] = "reported_daily"


fresno_area_express_ridership = fresno_area_express_ridership.rename(columns={
    'StopID': 'stop_id',
    'StopLabel': 'stop_name',
    'ProjectedBoarding': 'daily_boardings',
    'ProjectedAlighting': 'daily_alightings'
})

# Calculate average ridership per day type and stop
fresno_area_express_ridership['daily_boardings'] = pd.to_numeric(fresno_area_express_ridership['daily_boardings'], errors='coerce')
fresno_area_express_ridership['daily_alightings'] = pd.to_numeric(fresno_area_express_ridership['daily_alightings'], errors='coerce')

average_fresno_area_express_ridership = compute_averages(
    df=fresno_area_express_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["stop_id",  "stop_name", "day_type"]
)

average_fresno_area_express_ridership["organization_name"] = "Fresno County"

average_fresno_area_express_ridership[['start_date','end_date']] = [fresno_area_express_ridership['start_date'].min(), fresno_area_express_ridership['end_date'].max()]

fresno_final = standardize_columns(average_fresno_area_express_ridership, master_cols)

fresno_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
3571,Fresno County,1656,NE CHESTNUT - CHURCH,None,None,Sunday,27.591713,2.782598,2024-09-01,2025-08-31
3504,Fresno County,1633,SW MAPLE - VINE,None,None,Saturday,0.042721,0.145462,2024-09-01,2025-08-31
4015,Fresno County,1886,NW NEES - MILLBROOK,None,None,Sunday,1.759483,4.874653,2024-09-01,2025-08-31
4009,Fresno County,1883,NW TEAGUE - EIGHTH,None,None,Sunday,0.067429,0.000000,2024-09-01,2025-08-31
2753,Fresno County,1304,SW FIRST - HUNTINGTON,None,None,Weekday,5.394737,11.782540,2024-09-01,2025-08-31


In [19]:
gold_coast_transit_ridership = gold_coast_transit_ridership.rename(columns={
    'day_of_week': 'day_type',
    'lat': 'stop_lat',
    'lon': 'stop_lon'
})

group_cols = ["day_type", "stop_id", "stop_name",  "stop_lat", "stop_lon", "start_date", "end_date"]

# Sum AVG On 
total_gold_coast_transit_ridership = (gold_coast_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("total_on", "sum"),
        daily_alightings=("total_off", "sum"),
    )
)

total_gold_coast_transit_ridership['start_date'] = pd.to_datetime(total_gold_coast_transit_ridership['start_date'])
total_gold_coast_transit_ridership['end_date'] = pd.to_datetime(total_gold_coast_transit_ridership['end_date'])

total_gold_coast_transit_ridership["n_days"] = total_gold_coast_transit_ridership.apply(count_service_days, axis=1)

In [20]:
# Taking weighted average of the ridership across 4 different time periods
averaged_gold_coast_transit_ridership = (
    total_gold_coast_transit_ridership
    .groupby(['day_type', 'stop_name',
              'stop_id','stop_lat','stop_lon'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['daily_boardings'], weights=x['n_days']),
        'average_daily_alightings': np.average(x['daily_alightings'], weights=x['n_days'])
    }))
    .reset_index()
)

averaged_gold_coast_transit_ridership["organization_name"] = "Gold Coast Transit"

averaged_gold_coast_transit_ridership[['start_date','end_date']] = [gold_coast_transit_ridership['start_date'].min(), gold_coast_transit_ridership['end_date'].max()]
golden_coast_transit_final = standardize_columns(averaged_gold_coast_transit_ridership, master_cols)

golden_coast_transit_final.sample(5)

/tmp/ipykernel_3806/1215055102.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
4,Gold Coast Transit,1STMAR1,1st & Marguita,34.202734,-119.166086,Weekday,12.0,7.0,2025-05-01,2025-05-31
410,Gold Coast Transit,TELASH1,Telegraph & Ashwood,34.274443,-119.239258,Weekday,18.0,16.0,2025-05-01,2025-05-31
406,Gold Coast Transit,STURIC1,Sturgis & Rice,34.201766,-119.143858,Weekday,3.0,2.0,2025-05-01,2025-05-31
130,Gold Coast Transit,DORHST1,Doris & H,34.208082,-119.187906,Weekday,5.0,2.0,2025-05-01,2025-05-31
203,Gold Coast Transit,HUEJST2,Hueneme & J,34.147397,-119.186386,Weekday,19.0,29.0,2025-05-01,2025-05-31


In [21]:
# Add day_type from the date column
golden_gate_park_shuttle_ridership = add_day_type(golden_gate_park_shuttle_ridership, date_col="Date")

group_cols = [
    "Date", "day_type", "Stop", "Stop ID", "start_date", "end_date"]

# Sum AVG On 
total_golden_gate_park_shuttle_ridership = (golden_gate_park_shuttle_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("Ridership", "sum"),
    )
)

total_golden_gate_park_shuttle_ridership["daily_ridership_basis"] = "reported_daily"

total_golden_gate_park_shuttle_ridership = total_golden_gate_park_shuttle_ridership.rename(columns={
    'Stop': 'stop_name',
    'Date': 'date',
    'Stop ID': 'stop_id'
})

# Calculate average ridership per day type and stop
total_golden_gate_park_shuttle_ridership['daily_boardings'] = pd.to_numeric(total_golden_gate_park_shuttle_ridership['daily_boardings'], errors='coerce')
average_golden_gate_park_shuttle_ridership = compute_averages(
    df=total_golden_gate_park_shuttle_ridership,
    ridership_cols=["daily_boardings"],
    group_cols=["stop_name", "day_type", "stop_id"]
)


average_golden_gate_park_shuttle_ridership["organization_name"] = "Golden Gate Park Shuttle"

average_golden_gate_park_shuttle_ridership[['start_date','end_date']] = [golden_gate_park_shuttle_ridership['start_date'].min(), 
                                                                         golden_gate_park_shuttle_ridership['end_date'].max()]
golden_gate_parkshuttle_final = standardize_columns(average_golden_gate_park_shuttle_ridership, master_cols)

golden_gate_parkshuttle_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
53,Golden Gate Park Shuttle,7601,Transverse,None,None,Weekday,9.118774,None,2024-07-01,2025-06-30
22,Golden Gate Park Shuttle,7613,Conservatory of Flowers WB,None,None,Sunday,17.653846,None,2024-07-01,2025-06-30
19,Golden Gate Park Shuttle,7612,Conservatory of Flowers EB,None,None,Sunday,9.269231,None,2024-07-01,2025-06-30
46,Golden Gate Park Shuttle,7615,Tennis Center/ Dalia Dell EB,None,None,Sunday,8.730769,None,2024-07-01,2025-06-30
51,Golden Gate Park Shuttle,7601,Transverse,None,None,Saturday,10.826923,None,2024-07-01,2025-06-30


In [22]:
# Add day_type from the date column
golden_gate_transit_ridership = add_day_type(golden_gate_transit_ridership, date_col="date")

golden_gate_transit_ridership = golden_gate_transit_ridership.rename(columns={
    'STOP_NUMBER': 'stop_id',
    'STOP_NAME': 'stop_name',
})

# Remove everything in parentheses (including the parentheses) at the end of the string
golden_gate_transit_ridership['stop_name'] = golden_gate_transit_ridership['stop_name'].str.replace(r'\s*\(.*\)$', '', regex=True)

group_cols = [
    "day_type", "stop_id", "stop_name", "end_date", "start_date"]

# Sum AVG On and AVG Off
total_golden_gate_transit_ridership = (golden_gate_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("BOARDINGS", "sum"),
        daily_alightings=("ALIGHTINGS", "sum")
    )
)

total_golden_gate_transit_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_golden_gate_transit_ridership['daily_boardings'] = pd.to_numeric(total_golden_gate_transit_ridership['daily_boardings'], errors='coerce')
total_golden_gate_transit_ridership['daily_alightings'] = pd.to_numeric(total_golden_gate_transit_ridership['daily_alightings'], errors='coerce')

average_golden_gate_transit_ridership = compute_averages(
    df=total_golden_gate_transit_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["stop_id", "stop_name", "day_type"]
)

average_golden_gate_transit_ridership["organization_name"] = "Golden Gate Transit"
average_golden_gate_transit_ridership[['start_date','end_date']] = [golden_gate_transit_ridership['start_date'].min(), 
                                                                         golden_gate_transit_ridership['end_date'].max()]

golden_gate_transit_final = standardize_columns(average_golden_gate_transit_ridership, master_cols)

golden_gate_transit_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
49,Golden Gate Transit,40037,Golden Gate Bridge Toll Plaza,None,None,Sunday,139.25000,125.750000,2025-09-01,2025-09-30
704,Golden Gate Transit,80033,VTP 101 SB adj to Lincoln & Wilson,None,None,Saturday,0.00000,0.000000,2025-09-01,2025-09-30
296,Golden Gate Transit,40695,Redwood Blvd & Olive Ave,None,None,Sunday,4.75000,7.500000,2025-09-01,2025-09-30
326,Golden Gate Transit,40716,S Novato Blvd & Redwood Blvd,None,None,Weekday,12.00000,0.428571,2025-09-01,2025-09-30
348,Golden Gate Transit,40862,S McDowell Blvd & Baywood Dr,None,None,Weekday,0.47619,0.857143,2025-09-01,2025-09-30


In [23]:
long_beach_transit_ridership["daily_ridership_basis"] = "reported_avg_daily"

group_cols = [
    "DayType", "StopID", "StopName", "end_date", "start_date", "daily_ridership_basis"]

# Sum AVG On and AVG Off
total_long_beach_transit_ridership = (long_beach_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("Boardings", "sum"),
        average_daily_alightings=("Alightings", "sum")
    )
)

total_long_beach_transit_ridership = total_long_beach_transit_ridership.rename(columns={
    'StopID': 'stop_id',
    'StopName': 'stop_name',
    'DayType': 'day_type',
})

total_long_beach_transit_ridership["organization_name"] = "Long Beach Transit"

total_long_beach_transit_ridership[['start_date','end_date']] = [long_beach_transit_ridership['start_date'].min(), 
                                                                         long_beach_transit_ridership['end_date'].max()]

long_beach_transit_final = standardize_columns(total_long_beach_transit_ridership, master_cols)

long_beach_transit_final.sample(5)


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
2272,Long Beach Transit,531,Beach Dr & Dorms S,None,None,Sunday,35.697966,41.159028,2024-07-01,2025-06-30
652,Long Beach Transit,825,Orange & 33rd NE,None,None,Saturday,3.192793,6.380952,2024-07-01,2025-06-30
70,Long Beach Transit,89,Santa Fe & 25th NE,None,None,Saturday,96.671742,93.900048,2024-07-01,2025-06-30
2290,Long Beach Transit,557,Long Beach Blvd & 32nd NE,None,None,Sunday,0.000000,0.097525,2024-07-01,2025-06-30
4831,Long Beach Transit,1665,PCH & Martin Luther King Jr SW,None,None,Weekday,93.163085,102.931357,2024-07-01,2025-06-30


In [24]:
octa_ridership = octa_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',

})

octa_ridership["stop_name"] = octa_ridership["stop_name"].str.split("-", n=1).str[1]

group_cols = [
    "day_type", "stop_id", "stop_name", "end_date", "start_date"]

# Sum AVG On and AVG Off
octa_ridership_sum = (octa_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("APC Boarding", "sum"),
        average_daily_alightings=("APC Alighting", "sum")
    )
)

octa_ridership_sum.loc[
    octa_ridership_sum['day_type'].str.strip().str.lower() == "weekday",
    'day_type'
] = "Weekday"

octa_ridership_sum["organization_name"] = "Orange County Transportation Authority"

octa_ridership_sum[['start_date','end_date']] = [octa_ridership['start_date'].min(), 
                                                                         octa_ridership['end_date'].max()]

octa_final = standardize_columns(octa_ridership_sum, master_cols)

octa_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
2144,Orange County Transportation Authority,3378,IRVINE CENTER-SPECTRUM,None,None,Weekday,12,37,2025-02-04,2025-02-04
3025,Orange County Transportation Authority,5005,JAMBOREE-BRISTOL,None,None,Weekday,6,27,2025-02-04,2025-02-04
625,Orange County Transportation Authority,895,ORANGETHORPE-KNOTT,None,None,Weekday,0,4,2025-02-04,2025-02-04
3604,Orange County Transportation Authority,5980,1ST-DOWNTOWN PLAZA,None,None,Weekday,142,113,2025-02-04,2025-02-04
591,Orange County Transportation Authority,854,LA PALMA-EL MONTE,None,None,Weekday,8,27,2025-02-04,2025-02-04


In [25]:
omni_trans_ridership= omni_trans_ridership.rename(columns={
    'Stop Name': 'stop_name',
    'avg_boardings': 'average_daily_boardings',
    'avg_alightings': 'average_daily_alightings',
    'Stop Id': 'stop_id'
})

omni_trans_ridership["organization_name"] = "Omnitrans"

omni_trans_ridership_filtered = omni_trans_ridership[['stop_name', 'stop_id',  'average_daily_boardings', 'average_daily_alightings', 'organization_name', 'day_type']]

omni_trans_ridership_filtered['stop_id'] = omni_trans_ridership_filtered['stop_id'].apply(
    lambda x: None if pd.isna(x) else int(x)
)

omni_trans_ridership_filtered[['start_date','end_date']] = [omni_trans_ridership['start_date'].min(), 
                                                                         omni_trans_ridership['end_date'].max()]
omni_final = standardize_columns(omni_trans_ridership_filtered, master_cols)

omni_final.sample(5)

/tmp/ipykernel_3806/3069775595.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  omni_trans_ridership_filtered['stop_id'] = omni_trans_ridership_filtered['stop_id'].apply(
/tmp/ipykernel_3806/3069775595.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  omni_trans_ridership_filtered[['start_date','end_date']] = [omni_trans_ridership['start_date'].min(),
/tmp/ipykernel_3806/3069775595.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_in

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
4703,Omnitrans,7627.0,Francis @ Parco,None,None,all,0.232877,0.509589,2023-07-01,2026-06-30
1324,Omnitrans,8781.0,Chino Transit Center,None,None,all,30.265027,21.647541,2023-07-01,2026-06-30
866,Omnitrans,6660.0,Riverside @ Merrill,None,None,all,10.079235,9.368852,2023-07-01,2026-06-30
779,Omnitrans,8621.0,San Bernardino @ Orchard,None,None,all,1.013661,1.814208,2023-07-01,2026-06-30
3608,Omnitrans,5290.0,Highland @ Golden,None,None,all,0.843836,2.926027,2023-07-01,2026-06-30


In [26]:
riverside_transit_ridership = add_day_type(riverside_transit_ridership, date_col='date')

group_cols = [
    "date", "Stop ID", "start_date", 
    "end_date", "day_type"]

# Sum AVG On and AVG Off
total_riverside_transit_ridership = (riverside_transit_ridership.groupby(
    group_cols, as_index=False).agg(
    daily_boardings = ('ridership', 'sum'))
)

total_riverside_transit_ridership = total_riverside_transit_ridership.rename(columns={
    'Stop ID': 'stop_id'

})

In [27]:
total_riverside_transit_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_riverside_transit_ridership['daily_boardings'] = pd.to_numeric(total_riverside_transit_ridership['daily_boardings'], errors='coerce')
average_riverside_transit_ridership = compute_averages(
    df=total_riverside_transit_ridership,
    ridership_cols=["daily_boardings"],
    group_cols=["stop_id", "day_type"]
)

average_riverside_transit_ridership["organization_name"] = "Riverside Transit"

average_riverside_transit_ridership[['start_date','end_date']] = [riverside_transit_ridership['start_date'].min(), 
                                                                         riverside_transit_ridership['end_date'].max()]

riverside_final = standardize_columns(average_riverside_transit_ridership, master_cols)

riverside_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
2029,Riverside Transit,1790,None,None,None,Saturday,1.695652,None,2025-01-01,2025-10-31
6050,Riverside Transit,4698,None,None,None,Saturday,2.142857,None,2025-01-01,2025-10-31
3760,Riverside Transit,2493,None,None,None,Saturday,2.185185,None,2025-01-01,2025-10-31
5101,Riverside Transit,3261,None,None,None,Saturday,2.392857,None,2025-01-01,2025-10-31
999,Riverside Transit,1393,None,None,None,Weekday,7.576087,None,2025-01-01,2025-10-31


In [28]:
group_cols = [
    "stop_id", "stop_lat", "stop_lon", "stop_name", "day_type"]

# Combining across directions
sacrt_bus_ridership_sum = (sacrt_bus_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("average_boardings", "sum"),
        average_daily_alightings=("average_alightings", "sum")
    )
)

sacrt_bus_ridership_sum.loc[
    sacrt_bus_ridership_sum['day_type'].str.strip().str.lower() == "weekday",
    'day_type'
] = "Weekday"

# Replace day_type
sacrt_bus_ridership_sum["day_type"] = sacrt_bus_ridership_sum["day_type"].replace("daily", "all")

sacrt_bus_ridership_sum["organization_name"] = "SacRT Bus"

sacrt_bus_ridership_sum[['start_date','end_date']] = [sacrt_bus_ridership['start_date'].min(), 
                                                                         sacrt_bus_ridership['end_date'].max()]

sacrt_bus_final = standardize_columns(sacrt_bus_ridership_sum, master_cols)

sacrt_bus_final.sample(5)


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
2,SacRT Bus,105,I-5 FWY & SEAMAS AVE (NB),38.525752,-121.520596,Weekday,1,2,2023-09-01,2023-12-31
1567,SacRT Bus,2068,MEADOWVIEW RD & BRANCHWOOD WAY (WB),38.481533,-121.464043,all,2,3,2023-09-01,2023-12-31
2159,SacRT Bus,3066,WATT AVE & N HAVEN DR (SB),38.671074,-121.383064,all,24,11,2023-09-01,2023-12-31
1307,SacRT Bus,1717,J ST & 18TH ST (EB),38.577412,-121.482898,all,23,14,2023-09-01,2023-12-31
1351,SacRT Bus,1782,Q ST & 16TH ST (EB),38.570259,-121.488378,Weekday,0,0,2023-09-01,2023-12-31


In [29]:
samtrans_ridership = add_day_type(samtrans_ridership, date_col='APC Date')

samtrans_ridership = samtrans_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'Lat': 'stop_lat',
    'Lon': 'stop_lon',
    'APC Date': 'date',
    'Ons': 'daily_boardings',
    'Offs': 'daily_alightings',
})

samtrans_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
samtrans_ridership['daily_boardings'] = pd.to_numeric(samtrans_ridership['daily_boardings'], errors='coerce')
samtrans_ridership['daily_alightings'] = pd.to_numeric(samtrans_ridership['daily_alightings'], errors='coerce')

average_samtrans_ridership = compute_averages(
    df=samtrans_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["stop_id", "stop_name", "stop_lat", "stop_lon", "day_type"]
)

average_samtrans_ridership["organization_name"] = "Samtrans"

average_samtrans_ridership[['start_date','end_date']] = [samtrans_ridership['start_date'].min(), 
                                                                         samtrans_ridership['end_date'].max()]

samtrans_final = standardize_columns(average_samtrans_ridership, master_cols)

samtrans_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
79,Samtrans,311072,Hickey Blvd & Parkview Cir,37.657543,-122.478949,Saturday,0.000000,1.750000,2025-08-01,2025-08-31
2328,Samtrans,341028,E 3rd Ave & S Norfolk St,37.573890,-122.309944,Weekday,1.047619,11.285714,2025-08-01,2025-08-31
2251,Samtrans,340047,El Camino Real & Howard Ave,37.575893,-122.346321,Sunday,36.500000,26.000000,2025-08-01,2025-08-31
822,Samtrans,332110,Gellert Blvd & Hickey Blvd,37.665107,-122.469024,Sunday,8.800000,15.000000,2025-08-01,2025-08-31
3270,Samtrans,344164,Jefferson Ave & Ruby St,37.471183,-122.243476,Saturday,1.000000,0.000000,2025-08-01,2025-08-31


In [30]:
group_cols = [
    "Stop ID",  "Stop Name", "day_type", "start_date", 
    "end_date"]

# Sum AVG On and AVG Off
total_sdmts_ridership = (sdmts_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("Average On", "sum"),
        average_daily_alightings=("Average Off", "sum")
    )
)


total_sdmts_ridership = total_sdmts_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
})

total_sdmts_ridership["daily_ridership_basis"] = "reported_avg_daily"

total_sdmts_ridership["organization_name"] = "SDMTS"

total_sdmts_ridership[['start_date','end_date']] = [sdmts_ridership['start_date'].min(), 
                                                                         sdmts_ridership['end_date'].max()]

sdmts_final = standardize_columns(total_sdmts_ridership, master_cols)

sdmts_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
6070,SDMTS,40047,Jamacha Bl & San Diego St,None,None,Sunday,0.540441,8.102941,2024-09-01,2025-01-25
8995,SDMTS,75020,Santee Town Center Station,None,None,Sunday,1368.627381,1213.820238,2024-09-01,2025-01-25
6230,SDMTS,40194,Broadway & Ballantyne St,None,None,Weekday,55.432896,39.681657,2024-09-01,2025-01-25
3626,SDMTS,12239,Meadowbrook Dr & Skyline Dr,None,None,Weekday,13.389617,76.844696,2024-09-01,2025-01-25
205,SDMTS,10139,El Cajon Bl & Arizona St,None,None,Weekday,32.030513,48.667429,2024-09-01,2025-01-25


In [31]:
santa_cruz_metro_ridership = santa_cruz_metro_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'Boardings': 'average_daily_boardings',
    'Alightings': 'average_daily_alightings'    
})

santa_cruz_metro_ridership["organization_name"] = "Santa Cruz Metro"

santa_cruz_metro_ridership[['start_date','end_date']] = [santa_cruz_metro_ridership['start_date'].min(), 
                                                                         santa_cruz_metro_ridership['end_date'].max()]

santa_cruz_final = standardize_columns(santa_cruz_metro_ridership, master_cols)

santa_cruz_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
624,Santa Cruz Metro,1748,Scotts Valley Dr + Terrace View Dr,None,None,all,0.852055,2.328767,2024-07-01,2025-06-30
377,Santa Cruz Metro,1505,Heller Dr (UCSC - Oakes College),None,None,all,67.838356,74.284932,2024-07-01,2025-06-30
654,Santa Cruz Metro,1783,Soquel Ave + Ocean,None,None,all,43.183562,4.646575,2024-07-01,2025-06-30
172,Santa Cruz Metro,2479,Clifford Ave + Rosewood Dr,None,None,all,2.402740,1.243836,2024-07-01,2025-06-30
178,Santa Cruz Metro,1325,Clubhouse Dr + St Andrews Dr,None,None,all,0.024658,0.120548,2024-07-01,2025-06-30


In [32]:
sbmtd_ridership = sbmtd_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'avg_boardings': 'daily_boardings',
    'avg_ridership': 'daily_alightings'    
})


sbmtd_ridership["n_days"] = sbmtd_ridership.apply(count_service_days, axis=1)

# Taking weighted average of the ridership across 12 months
averaged_sbmtd_ridership = (
    sbmtd_ridership
    .groupby(['stop_id', 'stop_name', 'day_type'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['Ridership by Stop_Boardings'], weights=x['n_days']),
        'average_daily_alightings': np.average(x['Ridership by Stop_Alightings'], weights=x['n_days'])
    }))
    .reset_index()
)



averaged_sbmtd_ridership["organization_name"] = "SBMTD"

averaged_sbmtd_ridership[['start_date','end_date']] = [sbmtd_ridership['start_date'].min(), 
                                                                         sbmtd_ridership['end_date'].max()]

sbmtd_final = standardize_columns(averaged_sbmtd_ridership, master_cols)

sbmtd_final.sample(5)

/tmp/ipykernel_3806/1734702637.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
125,SBMTD,143,Hollister & San Marcos,None,None,all,238.608219,163.304110,2024-11-01,2025-10-31
238,SBMTD,285,Pesetas & Calle Real,None,None,all,387.378082,170.073973,2024-11-01,2025-10-31
137,SBMTD,172,Hollister & San Ricardo,None,None,all,232.106849,325.164384,2024-11-01,2025-10-31
444,SBMTD,531,La Cumbre & Pueblo,None,None,all,7.567123,6.375342,2024-11-01,2025-10-31
219,SBMTD,266,Hitchcock & State,None,None,all,30.794521,253.421918,2024-11-01,2025-10-31


In [33]:
burbank_ridership = burbank_ridership.rename(columns={
    'Organization Name ': 'organization_name',
    'StopName': 'stop_name',
    'Arrive': 'start_date', 
})


burbank_ridership['start_date'] = pd.to_datetime(
    burbank_ridership['start_date'], errors='coerce'
).dt.date

burbank_ridership['end_date'] = burbank_ridership['start_date']
burbank_ridership['end_date'] = pd.to_datetime(burbank_ridership['end_date'], errors='coerce')

burbank_ridership = add_day_type(burbank_ridership, date_col='start_date')

group_cols = [
    "day_type", "stop_name", "end_date", "start_date", "organization_name"]


# Sum AVG On and AVG Off
total_burbank_ridership = (burbank_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("Ons", "sum"),
        daily_alightings=("Offs", "sum")
    )
)

# Calculate average ridership per day type and stop
total_burbank_ridership['daily_boardings'] = pd.to_numeric(total_burbank_ridership['daily_boardings'], errors='coerce')
total_burbank_ridership['daily_alightings'] = pd.to_numeric(total_burbank_ridership['daily_alightings'], errors='coerce')

average_burbank_ridership = compute_averages(
    df=total_burbank_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["organization_name", "stop_name", "day_type"]
)

average_burbank_ridership[['start_date','end_date']] = [burbank_ridership['start_date'].min(), 
                                                                         burbank_ridership['end_date'].max()]
burbank_ridership_final = standardize_columns(average_burbank_ridership, master_cols)
burbank_ridership_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
18,City of Burbank,None,Empire and Niagara,None,None,Weekday,0.454545,1.227273,2024-05-01,2024-05-31
5,City of Burbank,None,Buena Vista & Olive SB,None,None,Weekday,4.000000,3.590909,2024-05-01,2024-05-31
21,City of Burbank,None,Hollywood & Olive EB,None,None,Weekday,3.681818,9.136364,2024-05-01,2024-05-31
4,City of Burbank,None,Buena Vista & Olive NB,None,None,Weekday,3.272727,3.090909,2024-05-01,2024-05-31
22,City of Burbank,None,Hollywood & Pacific,None,None,Weekday,0.727273,0.136364,2024-05-01,2024-05-31


In [35]:
all_ridership = pd.concat([
    bart_final,
    big_blue_bus_ridership_final,
    burbank_ridership_final,
    caltrain_ridership_final,
    culver_citybus_final,
    foothill_final,
    fresno_final,
    golden_coast_transit_final,
    golden_gate_parkshuttle_final,
    golden_gate_transit_final,
    long_beach_transit_final,
    octa_final,
    omni_final,
    riverside_final,
    sacrt_bus_final,
    samtrans_final,
    sdmts_final,
    santa_cruz_final,
    sbmtd_final,
], ignore_index=True)

/tmp/ipykernel_3806/2699512867.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_ridership = pd.concat([


In [36]:
all_ridership.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55566 entries, 0 to 55565
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   organization_name         55566 non-null  object        
 1   stop_id                   55111 non-null  object        
 2   stop_name                 44346 non-null  object        
 3   stop_lat                  15342 non-null  float64       
 4   stop_lon                  15342 non-null  float64       
 5   day_type                  55566 non-null  object        
 6   average_daily_boardings   55566 non-null  float64       
 7   average_daily_alightings  49184 non-null  float64       
 8   start_date                55566 non-null  datetime64[ns]
 9   end_date                  55566 non-null  datetime64[ns]
dtypes: datetime64[ns](2), float64(4), object(4)
memory usage: 4.2+ MB


In [37]:
all_ridership.to_csv(f"{GCS_FILE_PATH}/AHSC_2026/ridership_all.csv", index=False)